# SQL JOINs Explained Visually: INNER vs LEFT JOIN

Why did your query lose rows? In this lesson you'll see exactly how INNER JOIN and LEFT JOIN match rows between two tables, using one small college-events example from start to finish.

Part of the **SQL for Data & AI** series on NeuralNest.

Run the code cells from top to bottom. Change the numbers and run them again: that's the fastest way to really understand it.

## Why this matters

You run a query, and half your rows just vanish. No error, no warning. They're simply gone. By the end of this video, you'll know exactly where they went, and how to get them back.

## Why we need joins

- Data is split across tables
- A join matches rows by a key
- The wrong join silently drops rows

Real databases split data into separate tables, so nothing is stored twice. A join puts them back together by matching rows on a shared key, like an ID. But pick the wrong kind of join, and rows disappear without a single error message.

## Our two tables

**students**

| id | name | event_id |
|---|---|---|
| 1 | Asha | 10 |
| 2 | Ravi | NULL |
| 3 | Meera | 20 |
| 4 | Kabir | 10 |

**events**

| id | title |
|---|---|
| 10 | Hackathon |
| 20 | Quiz |
| 30 | Robotics |

Here's our example: a college fest. On the left, four students, and the event each one signed up for. Ravi hasn't picked an event yet, so his event ID is NULL. On the right, three events. Notice that nobody has signed up for Robotics.

## INNER JOIN: only matches

Two sets: **students** and **events**. Highlighted region: **inner**.

*Keeps rows that match on both sides*

Think of each table as a circle. An inner join keeps only the overlap: students whose event ID matches a real event. Anything without a partner on the other side is left out.

## Writing the INNER JOIN

Here's the query. We select each student's name and event title. The on clause says how rows match: the event's ID must equal the student's event ID. The letters s and e are just short aliases for the two tables.

```sql
SELECT s.name, e.title
FROM students s
INNER JOIN events e
  ON e.id = s.event_id;
```

## Who survives the INNER JOIN?

**students** (highlighted rows in **bold**)

| id | name | event_id |
|---|---|---|
| **1** | **Asha** | **10** |
| 2 | Ravi | NULL |
| **3** | **Meera** | **20** |
| **4** | **Kabir** | **10** |

Watch which students make it. The highlighted rows are what the query returns: Asha, Meera and Kabir. Ravi is gone, because NULL never equals anything, not even another NULL. Four students went in, three came out. That's where your missing rows went.

## LEFT JOIN: keep everyone

Two sets: **students** and **events**. Highlighted region: **left**.

*Every left row, matched or not*

Now the fix. A left join keeps the whole left circle: every student, whether they have a matching event or not. Just change the word inner to left in the query.

## The LEFT JOIN result

**result** (highlighted rows in **bold**)

| name | title |
|---|---|
| Asha | Hackathon |
| **Ravi** | **NULL** |
| Meera | Quiz |
| Kabir | Hackathon |

Now we get all four students. Ravi is back, and his event title is NULL, which simply means no match. That NULL is useful: it tells you exactly which students still need to choose an event.

## Try it yourself in Python

You can try this right now, with nothing to install. Python's built-in sqlite3 module creates a small database in memory. We create both tables, insert the same rows, and run the left join. It prints four, one row for every student.

In [ ]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE students(id, name, event_id);
CREATE TABLE events(id, title);
INSERT INTO students VALUES (1,'Asha',10),
  (2,'Ravi',NULL),(3,'Meera',20),(4,'Kabir',10);
INSERT INTO events VALUES (10,'Hackathon'),
  (20,'Quiz'),(30,'Robotics');
""")
q = """SELECT s.name, e.title FROM students s
LEFT JOIN events e ON e.id = s.event_id"""
print(len(db.execute(q).fetchall()))
# -> 4

## How many rows each join keeps

Here's the whole story in one chart. The inner join returns three rows. The left join returns four, every student. And a full outer join returns five, because it also keeps Robotics, the event nobody joined.

In [ ]:
import matplotlib.pyplot as plt

labels = ['INNER', 'LEFT', 'FULL']
width = 0.8 / 1
plt.bar([i - 0.4 + width * (0 + 0.5) for i in range(3)], [3.0, 4.0, 5.0], width, label='rows')
plt.xticks(range(3), labels)
plt.xlabel('join type')
plt.ylabel('rows returned')
plt.show()

## The classic LEFT JOIN mistake

- Filtering the right table in WHERE
- NULL rows fail the filter
- Put that condition in ON instead

One common mistake. You write a left join, then add a where clause on the events table, like title equals Hackathon. Ravi's title is NULL, so he fails that filter, and your left join quietly behaves like an inner join again. If you want to keep every student, move that condition into the on clause instead.

## Recap

- INNER JOIN keeps only matches
- LEFT JOIN keeps every left row
- NULL means no match
- Filter the right table in ON

Quick recap. Inner join keeps only matching rows. Left join keeps every row from the left table, with NULL where there's no match. And watch your where clause on a left join. Next in the series: self joins and anti joins, to find the rows that don't match at all.